# Введение
Коллаборативная фильтрация (КФ) – метод, используемый в рекомендательных системах, для прогнозирования неизвестных предпочтений одного пользователя по известным предпочтениям других пользователей. Но несмотря на всю пользу, КФ дает некоторые классические проблемы: матрица предпочтений почти полностью состоит из нулей и ее непросто хранить в памяти; для новых пользователей и видео есть проблема холодного старта, когда рекомендации не на чем строить; КФ хоть и несет задачу продвижения малоизвестных видео, но часто это работает наоборот. Наш датасет содержит данные о коротких видео, и на задачу рекомендательной системы для такого формата медиа можно смотреть как на создание гибрида, который оценивает вкусы пользователей и положение одного видео на карте смыслов, и который решает вышеперечисленные проблемы. Мы работаем с неявными сигналами, поэтому задействуем библиотеку implicit


# Цель работы
Исследовать влияние контентных эмбеддингов на качество рекомендаций, построенных методами коллаборативной фильтрации, на открытом датасете VK-LSVD: количественно оценить, улучшает ли добавление контентного сигнала точность и качество ранжирования рекомендаций, а также определить, в каких сценариях контентный сигнал наиболее полезен.

# Задачи работы
    1. Подготовить данные: агрегировать неявные поведенческие сигналы
    2. Реализовать несколько алгоритмов коллаборативной фильтрации
    3. Построить контентную модель на основе предоставленных датасетом мультимодальных эмбеддингов объектов (профиль пользователя как взвешенное среднее эмбеддингов просмотренного, косинусный скоринг).
    4. Реализовать гибридные модели — линейное смешивание оценок CF и контентной модели с подбором коэффициента α на валидации.
    5. Провести сравнение всех моделей по единому протоколу (Recall@10, Precision@10, NDCG@10 на temporal split), включая оценку устойчивости результатов (прогоны с различными случайными выборками, mean ± std) и сегментный анализ на малопопулярных объектах (cold-start).
    6. Сформулировать выводы о полезности контентного сигнала в различных сценариях; ввиду наличия в датасете единственного мультимодального эмбеддинга — исследовать доступные оси сравнения контентных сигналов (размерность эмбеддинга, контентный vs коллаборативный сигнал).


### Анализ четырёх типов контентных сигналов (текст / аудио / граф / мультимодальный)
Анализируем 4 типа контентных эмбеддингов. 3 из них берутся из датасета, собранный пользователями Twitter, когда они выкладывали сообщения о том что слушают прямо сейчас по тегу #nowplaying - nowplaying-RS (Zenodo).Мультимодальный изучается на датасете VK-LSVD, включающий кадры, аудио и описание шортса

| Сигнал | Датасет | Как получаем |
|---|---|---|
| Текстовый | nowplaying-RS: хэштеги слушателей | TF-IDF -> SVD |
| Аудио | nowplaying-RS: готовые характеристики Spotify | стандартизуем — вектор готов |
| Графовый | nowplaying-RS: граф пользователь–трек | item2vec (word2vec по сессиям) |
| Мультимодальный | **VK-LSVD: готовый эмбеддинг** (кадры+аудио+описание) | используем как есть, 32 компоненты |

Чтобы сравнить полезность каждого типа мы будем сравнивать прирост Recall от чистого CF к гибриду на хвосте непопулярных объектов. Используем алгоритм ALS

### 0. Установка библиотек и данных


Выбрали random_seed=42 

In [26]:
# Colab: ставим зависимости. implicit и gensim тянут C-расширения — установка ~1-2 мин
!pip install implicit polars scikit-learn gensim huggingface_hub


In [27]:
import urllib.request
import zipfile
import os

In [28]:
# Скачиваем архив датасета nowplaying-RS с Zenodo (CSV лежат внутри zip, ~1.3 ГБ)
#!wget "https://zenodo.org/records/3248543/files/nowplayingrs.zip?download=1" -O nowplayingrs.zip


In [29]:
url = "https://zenodo.org/records/3248543/files/nowplayingrs.zip?download=1"
zip_path = "nowplayingrs.zip"
urllib.request.urlretrieve(url, zip_path)
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(".")

In [35]:
#!unzip -o nowplayingrs.zip

In [3]:
# Ищем, куда распаковались CSV (архив может создать вложенную папку), и печатаем найденные пути
import glob, os
csv_paths = glob.glob('**/user_track_hashtag_timestamp.csv', recursive=True)
print('Найденные CSV:', csv_paths)
DATA_DIR = os.path.dirname(csv_paths[0]) if csv_paths else '.'
print('DATA_DIR =', DATA_DIR)


Найденные CSV: ['nowplaying_rs_dataset\\user_track_hashtag_timestamp.csv']
DATA_DIR = nowplaying_rs_dataset


In [5]:
# Импорты и фиксация случайности (для воспроизводимости)
import polars as pl
import numpy as np
import random
from scipy.sparse import coo_matrix
from implicit.als import AlternatingLeastSquares

RANDOM_SEED = 42
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)

## 1. Загрузка и сабсэмплинг

Подробнее о датасете:

**user_track_hashtag_timestamp.csv** - это данные об истории прослушивания. содержит данные идентификатора пользователя Twitter, идентификатора трека, хештег твита и дата публикации твита

**сontext_content_features.csv** - это аудио характеристики для каждого track_id, извлеченные через Spotify Web API. Для каждого идентификатора трека есть 9 характеристик от 0 до 1 (кроме loudness и tempo), описывающих песню

Использование полного датасета требует огромной траты памяти, поэтому мы возьмем репрезентативную подвыборку активных пользователей и треков: берем пользователей у которых не менее 20 событий, треки у которых не менее 10 событий, то есть активных пользователей и популярные треки для облегчения задачи обучения коллаборативному алгоритму.

Берем первые 10000 событий

In [8]:
# Читаем события (история прослушиваний) и аудио-характеристики, печатаем их колонки
events = pl.read_csv(
    f'{DATA_DIR}/user_track_hashtag_timestamp.csv',
    columns=['user_id', 'track_id', 'hashtag', 'created_at']
).with_columns(
    pl.col('created_at').str.strptime(pl.Datetime, format='%Y-%m-%d %H:%M:%S', strict=False)
)
print('events columns:', events.columns, '| строк:', len(events))

AUDIO_COLS = ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness',
              'instrumentalness', 'liveness', 'valence', 'tempo']
ctx = pl.read_csv(f'{DATA_DIR}/context_content_features.csv',
                  columns=['track_id'] + AUDIO_COLS,
                  infer_schema_length=10000, ignore_errors=True)
print('ctx columns:', ctx.columns)

# --- сабсэмплинг: активные пользователи и треки (иначе матрица слишком разрежена) ---
MIN_USER_EVENTS, MIN_TRACK_EVENTS = 20, 10
u_cnt = events.group_by('user_id').len()
t_cnt = events.group_by('track_id').len()
active_users  = u_cnt.filter(pl.col('len') >= MIN_USER_EVENTS)['user_id']
active_tracks = t_cnt.filter(pl.col('len') >= MIN_TRACK_EVENTS)['track_id']
events = events.filter(pl.col('user_id').is_in(active_users) &
                       pl.col('track_id').is_in(active_tracks))
print('после фильтра:', len(events), 'событий,',
      events['user_id'].n_unique(), 'пользователей,',
      events['track_id'].n_unique(), 'треков')

events columns: ['user_id', 'track_id', 'hashtag', 'created_at'] | строк: 17560113
ctx columns: ['instrumentalness', 'liveness', 'speechiness', 'danceability', 'valence', 'loudness', 'tempo', 'acousticness', 'energy', 'track_id']


C:\Temp\ipykernel_13556\2290360953.py:23: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  events = events.filter(pl.col('user_id').is_in(active_users) &


после фильтра: 16415566 событий, 36055 пользователей, 135390 треков


In [9]:
events

user_id,track_id,hashtag,created_at
i64,str,str,datetime[μs]
81496937,"""cd52b3e5b51da29e5893dba82a418a…","""nowplaying""",2014-01-01 05:54:21
81496937,"""cd52b3e5b51da29e5893dba82a418a…","""goth""",2014-01-01 05:54:21
81496937,"""cd52b3e5b51da29e5893dba82a418a…","""deathrock""",2014-01-01 05:54:21
81496937,"""cd52b3e5b51da29e5893dba82a418a…","""postpunk""",2014-01-01 05:54:21
2205686924,"""da3110a77b724072b08f231c9d6f75…","""NowPlaying""",2014-01-01 05:54:22
…,…,…,…
2819332208,"""03498f305040835c5f76d7c5660204…","""nowplaying""",2014-12-23 07:21:04
154070865,"""8bacefe018a221d933529dd466e7c1…","""nowplaying""",2014-12-23 07:21:07
985591650,"""0e64c11b9a77e93f343f9c1c0cdbcf…","""nowplaying""",2014-12-23 07:21:08


In [12]:
ctx

instrumentalness,liveness,speechiness,danceability,valence,loudness,tempo,acousticness,energy,track_id
f64,f64,f64,f64,f64,f64,f64,f64,f64,str
0.00479,0.18,0.0294,0.634,0.342,-8.345,125.044,0.00035,0.697,"""cd52b3e5b51da29e5893dba82a418a…"
0.0177,0.0638,0.0624,0.769,0.752,-8.252,95.862,0.267,0.826,"""da3110a77b724072b08f231c9d6f75…"
0.0,0.086,0.0436,0.675,0.775,-4.432,97.03,0.217,0.885,"""ba84d88c10fb0e42d4754a27ead105…"
0.0,0.143,0.0292,0.324,0.333,-5.647,74.101,0.239,0.574,"""33f95122281f76e7134f9cbea3be98…"
0.000183,0.362,0.0524,0.767,0.808,-5.011,114.237,0.0364,0.739,"""b5c42e81e15cd54b9b0ee34711dedf…"
…,…,…,…,…,…,…,…,…,…
0.0,0.122,0.138,0.758,0.914,-14.741,119.507,0.347,0.477,"""da58ba1ca758809bb86aa8e3a36d1e…"
0.000107,0.168,0.0291,0.626,0.36,-7.006,132.349,0.664,0.526,"""03498f305040835c5f76d7c5660204…"
0.0,0.0892,0.0378,0.375,0.34,-12.952,62.777,0.932,0.115,"""8bacefe018a221d933529dd466e7c1…"


## 2. Temporal split и матрица взаимодействий

Для чистоты обучения модели мы разрежем данные по хронологии train -> validation как прошлое и будущее, в процентном соотношении 90 на 10 по временной линии

отдельно считаем квантиль для колонки временного типа datetime[μs]

In [15]:
events = events.sort('created_at')
n = len(events)
k = int(n * 0.9)
split_ts = events['created_at'][k]  # 90-й процентиль по порядку
print('split_ts:', split_ts)
print('min date:', events['created_at'].min())
print('max date:', events['created_at'].max())

train_ev = events.filter(pl.col('created_at') <  split_ts)
val_ev   = events.filter(pl.col('created_at') >= split_ts)
print(f'train: {len(train_ev)}, validation: {len(val_ev)}')

split_ts: 2014-11-24 09:07:17
min date: 2014-01-01 05:54:21
max date: 2014-12-23 07:21:11
train: 14774009, validation: 1641557


In [17]:
train_ev

user_id,track_id,hashtag,created_at
i64,str,str,datetime[μs]
81496937,"""cd52b3e5b51da29e5893dba82a418a…","""nowplaying""",2014-01-01 05:54:21
81496937,"""cd52b3e5b51da29e5893dba82a418a…","""goth""",2014-01-01 05:54:21
81496937,"""cd52b3e5b51da29e5893dba82a418a…","""deathrock""",2014-01-01 05:54:21
81496937,"""cd52b3e5b51da29e5893dba82a418a…","""postpunk""",2014-01-01 05:54:21
2205686924,"""da3110a77b724072b08f231c9d6f75…","""NowPlaying""",2014-01-01 05:54:22
…,…,…,…
65086276,"""d1bf1d81b838548c90994380ccd09f…","""listenlive""",2014-11-24 09:07:13
1408949696,"""d8773dde6e57aef6bcf14284552b46…","""nowplaying""",2014-11-24 09:07:14
1408949696,"""d8773dde6e57aef6bcf14284552b46…","""listenlive""",2014-11-24 09:07:14


In [19]:
val_ev

user_id,track_id,hashtag,created_at
i64,str,str,datetime[μs]
164067322,"""df24e6b596f78263a81d96831ebee1…","""nowplaying""",2014-11-24 09:07:17
594937768,"""1c988c515296700ded81ec3edee54d…","""NowPlaying""",2014-11-24 09:07:20
594937768,"""1c988c515296700ded81ec3edee54d…","""Android""",2014-11-24 09:07:20
310835365,"""a633cd68bbfe361a9a959d47926861…","""NowPlaying""",2014-11-24 09:07:25
2732068658,"""5cd9e76c97ac7797e7161dc3431ea6…","""NowPlaying""",2014-11-24 09:07:25
…,…,…,…
2819332208,"""03498f305040835c5f76d7c5660204…","""nowplaying""",2014-12-23 07:21:04
154070865,"""8bacefe018a221d933529dd466e7c1…","""nowplaying""",2014-12-23 07:21:07
985591650,"""0e64c11b9a77e93f343f9c1c0cdbcf…","""nowplaying""",2014-12-23 07:21:08


Далее построим sparse_matrix которую будем факторизировать. Посчитаем веса взаимодействия по формуле расчета уровня уверенности в рек. системах с логарифмической модификацией (предложенный в статье Hu, Koren, Volinsky (2008) "Collaborative Filtering for Implicit Feedback Datasets").

c_ui = 1 + α * log(1 + r_ui) где r_ui это число взаимодействий

Данная модификация позволит создать ощутимую разницу между малопрослушиваемыми и частопрослушиваемыми треками так как в бинарных предпочтениях эти оба трека будут восприниматься с одинаковой силой = 1. Но также логарифм сглаживает эту разницу как если бы мы оставили просто x чтобы избежать переобучения.

поэтому считаем логарифмический вес = ln(1+x) методом .log1p() (x - число прослушиваний трека пользователем)

In [22]:
# считаем кол-во прослушиваний и логарифм
train_w = (train_ev.group_by(['user_id', 'track_id']).len()
                    .with_columns((1.0 + pl.col('len').log1p()).alias('weight')))

train_w

user_id,track_id,len,weight
i64,str,u32,f64
279655634,"""6e8dcf72468c4f9cf058c8fd0c7f4d…",1,1.693147
26837577,"""f113ab7b26872693b16d476a20c1d5…",1,1.693147
87050217,"""d4cbd1c0d6cd7d2a25522defc0827f…",5,2.791759
492455721,"""0ec24d4e6b2e8edf22ce2de1a62c2c…",3,2.386294
2154026963,"""bc280920a4058f07f6d360ca584119…",1,1.693147
…,…,…,…
1214078629,"""5f7f6a5aedc760545613d43cab71a2…",1,1.693147
1550121260,"""a86b5ba07cd7f40b6bb26847bc7192…",3,2.386294
1406517488,"""544cd55e2c9fab960d2edb49e4f0be…",1,1.693147


Коллаборативные модели, например ALS, работают с индексами, поэтому необходимо перевести все исходные идентификаторы в индексы столбцов и строк

In [25]:
# train_w то же что и train_ev только сгруппированный
# разницы в использовании нет если ищем уникальные значения
user_ids = sorted(train_w['user_id'].unique().to_list())
item_ids = sorted(train_w['track_id'].unique().to_list())
user_to_idx = {u: i for i, u in enumerate(user_ids)}
item_to_idx = {t: i for i, t in enumerate(item_ids)}

Собираем разреженную матрицу размером len(user_ids) X len(item_ids) с весами в ячейках, помня что мы заменили каждый user_id и item_id на их индексы
Используем конструктор формата COO, потом преобразуем в удобный формат CSR для операций над строками.
Удалим нули полученные при группировке

In [28]:
rows = train_w['user_id'].replace(user_to_idx).to_numpy()
cols = train_w['track_id'].replace(item_to_idx).to_numpy()
data = train_w['weight'].to_numpy()
sparse_matrix = coo_matrix((data, (rows, cols)),
                           shape=(len(user_ids), len(item_ids))).tocsr()
sparse_matrix.eliminate_zeros()

Проверим что матрица действительно покрывает всех пользователей и треки

Далее будем работать с функцией оценки. Подготовим val_ev и убираем дубликаты

In [31]:
assert sparse_matrix.shape == (len(user_ids), len(item_ids))
print('Matrix:', sparse_matrix.shape, 'nnz:', sparse_matrix.nnz)

val_interactions = val_ev.select(['user_id', 'track_id']).rename({'track_id': 'item_id'}).unique()

Matrix: (35772, 134685) nnz: 2058154


## 3. Строим четыре сигнала

### 3.1 Аудио — уже готов

Теперь необходимо получить вектор из 9 чисел для каждого трека. В датасете ctx один трек может встретиться много раз при повторном прослушивании, поэтому мы усредняем характеристики по каждому событию, чтобы получить один вектор, даже если в какой то момент характеристики слегка поменялись. is_in необходим чтобы получить строго те треки, которые встречаются в train_w.


In [34]:
audio_by_track = (ctx.filter(pl.col('track_id').is_in(item_ids))
                     .group_by('track_id').mean())
audio_by_track

track_id,instrumentalness,liveness,speechiness,danceability,valence,loudness,tempo,acousticness,energy
str,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""f61b5ceb67a141c43315fd8c850c91…",0.28,0.0633,0.185,0.401,0.207,-11.598,179.984,0.911,0.777
"""d737c1d5a860d08d4f14f6c6295749…",0.0341,0.0872,0.165,0.224,0.122,-4.38,173.54,0.000047,0.981
"""424f38a743b6551cfb5df19106784c…",0.000001,0.185,0.0412,0.406,0.31,-4.569,139.256,0.116,0.727
"""71dd36d94a93a297ef53ea66315e61…",0.0000172,0.118,0.124,0.479,0.495,-11.451,79.593,0.457,0.416
"""1a8eed62955a841719ce398ab2e2ef…",0.0,0.0749,0.028,0.803,0.406,-5.925,94.014,0.0858,0.358
…,…,…,…,…,…,…,…,…,…
"""0d5b877f9fb3bb4c4ba2dd5617ee5f…",0.205,0.3,0.0352,0.573,0.517,-8.127,171.854,0.208,0.62
"""7aed2e69e79c4d284acddb6d0ed141…",0.165,0.148,0.0636,0.441,0.38,-4.753,130.965,0.0568,0.811
"""2473717e80337a8b34cc944714924e…",0.906,0.137,0.0477,0.627,0.252,-8.351,126.005,0.000943,0.843


Построим словарь track_id: вектор

In [36]:
audio_map = {r['track_id']: [r[c] for c in AUDIO_COLS]
             for r in audio_by_track.iter_rows(named=True)}

Заполняем матрицу признаков. Создаем нулевую матрицу len(item_ids) X len(AUDIO_COLS), кладем в нее векторы в строку в соответствии с треком

In [38]:
audio_matrix = np.zeros((len(item_ids), len(AUDIO_COLS)), dtype=np.float32)
for iid, idx in item_to_idx.items():
    if iid in audio_map:
        audio_matrix[idx] = audio_map[iid]
audio_matrix

array([[ 4.48000e-01,  5.56000e-01, -9.00100e+00, ...,  9.17000e-02,
         3.06000e-01,  8.68900e+01],
       [ 6.96000e-01,  4.96000e-01, -1.23240e+01, ...,  7.14000e-02,
         3.11000e-01,  1.59973e+02],
       [ 4.69000e-01,  4.42000e-01, -9.92000e+00, ...,  6.16000e-02,
         4.36000e-01,  7.64180e+01],
       ...,
       [ 4.01000e-01,  1.69000e-01, -1.47820e+01, ...,  9.73000e-02,
         1.53000e-01,  1.20384e+02],
       [ 6.53000e-01,  5.98000e-01, -1.25690e+01, ...,  3.64000e-01,
         7.35000e-01,  9.76680e+01],
       [ 5.03000e-01,  9.29000e-01, -1.09870e+01, ...,  9.25000e-02,
         1.91000e-01,  1.08366e+02]], dtype=float32)

Наконец, этап стандартизации, необходимо для вычисления косинусного сходства
Слагаемое 1e-9 мы добавили в случае стандартного отклонения audio_matrix.std(0)=0 чтобы случайно не поделить среднее значение на 0

In [43]:
# z-score стандартизация по столбцам
audio_matrix = (audio_matrix - audio_matrix.mean(0)) / (audio_matrix.std(0) + 1e-9)
print('audio_matrix:', audio_matrix.shape)

audio_matrix: (134685, 9)


Теперь у нас есть матрица контентных признаков на основе аудио-характеристик, которую можно добавлять к CF моделям.

### 3.2 Текст — из хэштегов

Получим вектор из текстовых сигналов. Этими сигналами являются хештеги в твитах, описывающих трек.
Мы берем train_ev чтобы пройтись по каждому хэштегу. Использование train_w привело бы к пропуску твитов с разными тегами.

Фильтрация по ненулевым хэштегам и группировка по track_id. Конкатенируем хэштеги в одну строку. Далее все как с аудио-векторами: создаем словарь track_id: строка; строим список строк corpus.

In [47]:
# ========== 3.2 ТЕКСТ-сигнал: хэштеги -> TF-IDF -> SVD ==========
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# склеиваем все хэштеги трека в один "документ"
tag_docs = (train_ev.filter(pl.col('hashtag').is_not_null())
                    .group_by('track_id')
                    .agg(pl.col('hashtag').cast(pl.Utf8).str.join(' ').alias('doc')))
doc_map = dict(zip(tag_docs['track_id'].to_list(), tag_docs['doc'].to_list()))
corpus = [doc_map.get(iid, '') for iid in item_ids]     # строго в порядке item_to_idx
print('пример документа:', corpus[0][:120] if corpus else '(пусто)')

пример документа: NowPlaying NowPlaying NowPlaying nowplaying rock pops 70s nowplaying NP NowPlaying Music NP NowPlaying Music


Далее используя corpus создаем метрику TF-IDF для оценки релевантности и взвешивания каждого хэштега. Поставим ограничение в 5000 тегов и минимум 3 повтора на каждый тег

In [58]:
tfidf = TfidfVectorizer(max_features=5000, min_df=3)
X_tfidf = tfidf.fit_transform(corpus)
X_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 605167 stored elements and shape (134685, 5000)>

Используя Truncated SVD мы найдем латентные факторы, которые можно представить в виде 64 чисел. Таким образом мы получим текстовый эмбеддинг

In [62]:
text_matrix = TruncatedSVD(n_components=64, random_state=RANDOM_SEED)\
                 .fit_transform(X_tfidf).astype(np.float32)
print('text_matrix:', text_matrix.shape)

text_matrix: (134685, 64)


Теперь из хэштегов получены числовые векторы для каждого трека — это текстовый сигнал.

### 3.3 Графовый сигнал — item2vec (обучаем сами)

Графовый эмбеддинг кодирует соседство в графе прослушиваний: треки, которые слушают одни и те же пользователи подряд, получают близкие векторы. Реализуем упрощённый DeepWalk, который называют item2vec: имеем последовательность треков одного пользователя по времени, и по этим последовательностям обучаем обычный word2vec (skip-gram)

In [65]:
#%pip install --upgrade --only-binary :all: gensim

^C
Note: you may need to restart the kernel to use updated packages.


In [67]:
# 3.3 ГРаф-сигнал: item2vec
from gensim.models import Word2Vec

sentences = (train_ev.sort('created_at')
                     .group_by('user_id')
                     .agg(pl.col('track_id').cast(pl.Utf8).alias('seq'))
             )['seq'].to_list()
w2v = Word2Vec(sentences=sentences, vector_size=64, window=5,
               min_count=1, sg=1, epochs=5, seed=RANDOM_SEED, workers=4)

graph_matrix = np.zeros((len(item_ids), 64), dtype=np.float32)
for iid, idx in item_to_idx.items():
    key = str(iid)
    if key in w2v.wv:
        graph_matrix[idx] = w2v.wv[key]
print('graph_matrix:', graph_matrix.shape)

graph_matrix: (134685, 64)


### 3.4 Мультимодальный — готовый эмбеддинг VK-LSVD

В датасете VK-LSVD есть встроенный готовый эмбеддинг, который описывается как мультимодальный:

> Эмбеддинги обучаются исключительно на контенте (видео/описание/аудио и т. д.) — никаких дополнительных сигналов не добавляется.

Загрузим сабсэмпл и построим матрицу эмбеддингов из item_embeddings.npz

In [70]:
# ========== 3.4 МУЛЬТИМОДАЛЬНЫЙ сигнал: готовый эмбеддинг VK-LSVD ==========
from huggingface_hub import hf_hub_download

SUB = 'up0.001_ip0.001'
vk_files = ([f'subsamples/{SUB}/train/week_{i:02}.parquet' for i in range(25)]
            + [f'subsamples/{SUB}/validation/week_25.parquet', 'metadata/item_embeddings.npz'])
for f_ in vk_files:
    hf_hub_download(repo_id='deepvk/VK-LSVD', repo_type='dataset', filename=f_, local_dir='VK-LSVD')


In [72]:
vk_train = pl.concat([pl.scan_parquet(f'VK-LSVD/subsamples/{SUB}/train/week_{i:02}.parquet')
                      for i in range(25)]).collect(engine='streaming')
vk_val_interactions = (pl.read_parquet(f'VK-LSVD/subsamples/{SUB}/validation/week_25.parquet')
                         .select(['user_id', 'item_id']).unique())

In [74]:
# ВМЕСТО np.load(...) + np.stack([...])
emb_npz = np.load('VK-LSVD/metadata/item_embeddings.npz', mmap_mode='r')  # не грузит в RAM
emb_ids = emb_npz['item_id']                      # только id (лёгкий)
emb_pos = {int(t): k for k, t in enumerate(emb_ids)}
vk_items = sorted(set(vk_train['item_id'].unique().to_list()) & set(emb_pos.keys()))
vk_users = sorted(vk_train['user_id'].unique().to_list())
vk_user_to_idx = {u: i for i, u in enumerate(vk_users)}
vk_item_to_idx = {t: i for i, t in enumerate(vk_items)}

# читаем ТОЛЬКО нужные строки эмбеддинга (тысячи, а не 19 млн)
wanted_rows = np.array([emb_pos[t] for t in vk_items])
emb_full = emb_npz['embedding']                   # mmap-массив на диске
multi_matrix = np.asarray(emb_full[wanted_rows, :32]).astype(np.float32)
del emb_full, emb_npz                             # закрываем mmap

Веса взаимодействия строим по аналогии с train_w но также добавляем коэффициенты. Считаем что репост и закладка имеют большее значение чем иные типы взаимодействия, лайк и клик на автора как средняя реакция, открытие комментариев как неоднозначный показатель, и дизлайк как отрицательный показатель. Формула важна потому что социальные сети используют множество сигналов для построения рекомендательных алгоритмов, и при приравнии этих сигналов как к обычному просмотру единицы медиа, можно прийти к алгоритмическим абсурдам. Допустим в этой статье https://knightcolumbia.org/content/understanding-social-media-recommendation-algorithms описывается формула MSI, и согласно исследованию Facebook, взаимодействия, требующие больших усилий и направленные на других людей, воспринимаются как более осмысленные. И согласно статье, коэффициенты менялись сознательно вручную исследователями. ППоэтому имеет смысл делать такую же тактику

Общий вес обрезается до 100 чтобы предотвратить большие помехи, фильтруется от негативных весов


In [76]:
# веса VK-LSVD: 1 + log(время) + бонусы действий (та же логика, что в основной части)
vk_w = (vk_train.filter(pl.col('item_id').is_in(vk_items))
        .with_columns((1.0 + ((pl.col('timespent') / 60).clip(0, 60) + 1.0).log()
                        + pl.col('like').cast(pl.Int32) * 3.0
                        + pl.col('share').cast(pl.Int32) * 2.5
                        + pl.col('bookmark').cast(pl.Int32) * 2.0
                        + pl.col('click_on_author').cast(pl.Int32) * 1.5
                        + pl.col('open_comments').cast(pl.Int32) * 1.0
                        + pl.col('dislike').cast(pl.Int32) * (-2.0)
                       ).clip(0, 100).alias('weight'))
        .filter(pl.col('weight') > 0))

В результате получаем разреженную матрицу для ALS vk_matrix и словарь сигналов signal_matrices

In [78]:
r_ = vk_w['user_id'].replace(vk_user_to_idx).to_numpy()
c_ = vk_w['item_id'].replace(vk_item_to_idx).to_numpy()
vk_matrix = coo_matrix((vk_w['weight'].to_numpy(), (r_, c_)),
                       shape=(len(vk_users), len(vk_items))).tocsr()
vk_matrix.eliminate_zeros()
print('VK-LSVD matrix:', vk_matrix.shape, 'nnz:', vk_matrix.nnz)
print('multi_matrix (VK-LSVD embedding):', multi_matrix.shape)

VK-LSVD matrix: (10000, 19628) nnz: 47052060
multi_matrix (VK-LSVD embedding): (19628, 32)


In [79]:
# три сигнала одного каталога (nowplaying-RS); мультимодальный живёт на VK-LSVD отдельно
signal_matrices = {'text': text_matrix, 'audio': audio_matrix, 'graph': graph_matrix}

## 4. Модели: ALS-baseline и контент-модель на каждом сигнале

Обучаем и оцениваем все модели, которые будем сравнивать

Строим метод возвращающий массив k лучших индексов по убыванию score, который получает из scores на входе

Строим метод для вычисления Recall@K и NDCG@K для функции расчета score score_fn

Для каждого из 500 случайно отобранных пользователей вычисляем скоры всех айтемов через переданную score_fn, исключаем те, что уже были в train (через sparse_matrix[ui].indices), и оставляем топ‑K рекомендаций. Потом сравниваем их с реальными взаимодействиями из валидации, которые предварительно фильтруются: оставляем только пары, где и пользователь и айтем есть в train. Если задан item_idx_filter (для оценки холодного старта), то релевантные айтемы дополнительно фильтруются.

Recall@K = доля найденных релевантных среди топ‑K (нормированная на min(K, число релевантных))

NDCG@K = дисконтированная сумма позиций найденных релевантных, делённая на идеальную DCG. Пользователи, у которых после фильтрации не осталось релевантных айтемов, пропускаются.

In [85]:
#
def recommend_top_k_from_scores(scores, seen_idx, k=10):
    scores = scores.copy()
    if len(seen_idx):
        scores[np.array(list(seen_idx))] = -np.inf
    top = np.argpartition(-scores, k)[:k]
    return top[np.argsort(-scores[top])]

In [87]:
def evaluate_score_fn(score_fn, val_interactions, sparse_matrix, user_to_idx, item_to_idx,
                      K=10, sample_size=500, seed=42, item_idx_filter=None):
    rnd = random.Random(seed)
    vf = val_interactions.filter(pl.col('user_id').is_in(list(user_to_idx.keys())) &
                                 pl.col('item_id').is_in(list(item_to_idx.keys())))
    vu = vf['user_id'].unique().to_list()
    if len(vu) > sample_size:
        vu = rnd.sample(vu, sample_size)
    recalls, ndcgs = [], []
    for uid in vu:
        ui = user_to_idx[uid]
        true_idx = {item_to_idx[i] for i in
                    vf.filter(pl.col('user_id') == uid)['item_id'].to_list() if i in item_to_idx}
        if item_idx_filter is not None:
            true_idx &= item_idx_filter
        if not true_idx:
            continue
        rec = recommend_top_k_from_scores(score_fn(ui), sparse_matrix[ui].indices, k=K)
        hits = len(set(rec) & true_idx)
        recalls.append(hits / min(K, len(true_idx)))
        dcg = sum(1/np.log2(p+2) for p, r in enumerate(rec) if r in true_idx)
        idcg = sum(1/np.log2(p+2) for p in range(min(len(true_idx), K)))
        ndcgs.append(dcg/idcg if idcg > 0 else 0.0)
    return {'Recall@K': float(np.mean(recalls)) if recalls else 0,
            'NDCG@K': float(np.mean(ndcgs)) if ndcgs else 0,
            'Tested users': len(recalls)}

Обучение модели на датасете nowplaying-rs

In [105]:
import implicit.utils
implicit.utils._checked_blas_config = True

In [107]:
# ========== 4.2 ALS-baseline (nowplaying-RS) ==========
model = AlternatingLeastSquares(factors=64, regularization=0.1,
                                iterations=20, alpha=1.0, random_state=RANDOM_SEED)
model.fit(sparse_matrix)
print('user_factors:', model.user_factors.shape, '| item_factors:', model.item_factors.shape)

  0%|          | 0/20 [00:00<?, ?it/s]

user_factors: (35772, 64) | item_factors: (134685, 64)


Строим функцию скоринга для ALS. Результат -  скалярные произведения между вектором пользователя и всеми векторами айтемо. В получившемся массиве чем выше значения, тем более релевантен айтем

In [110]:
def als_scores(ui):
    return model.item_factors @ model.user_factors[ui]
print('ALS:', evaluate_score_fn(als_scores, val_interactions, sparse_matrix, user_to_idx, item_to_idx))

ALS: {'Recall@K': 0.02409126984126984, 'NDCG@K': 0.021998084995401818, 'Tested users': 500}


Для каждого пользователя вычисляется профиль как среднее арифметическое контентных векторов всех айтемов, с которыми он взаимодействовал в train (с учётом весов), а затем скор для любого айтема определяется как косинусное сходство между этим профилем и контентным вектором айтема. Никакие коллаборативные факторы не используются — модель опирается исключительно на близость контентных эмбеддингов. Это даёт базовый уровень, с которым сравниваются коллаборативные и гибридные подходы.

In [113]:
# ========== 4.3 Контент-модель и гибрид для ЛЮБОГО сигнала и ЛЮБОГО датасета ==========
def make_content_scores(content_matrix, inter_matrix):
    ws = inter_matrix @ content_matrix
    wt = np.asarray(inter_matrix.sum(axis=1)).flatten(); wt[wt == 0] = 1.0
    up = ws / wt[:, None]                       # профиль пользователя = взвеш. среднее эмбеддингов
    cn = np.linalg.norm(content_matrix, axis=1) + 1e-9
    def cs(ui, cm=content_matrix, up=up, cn=cn):
        p = up[ui]
        return (cm @ p) / (cn * (np.linalg.norm(p) + 1e-9))   # косинус профиля со всеми айтемами
    return cs

Функция min_max приводит значения к диапазону [0,1]

In [116]:
def min_max(x):
    lo, hi = x.min(), x.max()
    return (x - lo) / (hi - lo) if hi > lo else np.zeros_like(x)

Скоры от коллаборативной и контентной моделей по отдельности нормализуются в [0,1] через минимакс, а затем смешиваются с коэффициентом alpha (1 — чистый коллаб, 0 — чистый контент, промежуточные значения дают гибрид). Нормализация происходит отдельно для каждого пользователя, что вносит небольшие колебания масштаба между пользователями, но на практике допустимо и не мешает сравнению сигналов.



In [119]:
def make_hybrid(cf_fn, content_fn, alpha):
    def h(ui):
        return alpha * min_max(cf_fn(ui)) + (1 - alpha) * min_max(content_fn(ui))
    return h

## 5. Главный эксперимент: сравнение четырёх сигналов

Для каждого сигнала создаем
1) контент-модель соло
2) лучший гибрид с ALS (α по сетке)
3) гибрид на «хвосте», состоящий из непопулярных треков

In [122]:
# Сравнение сигналов (nowplaying-RS и мультимодальный на VK-LSVD)
def run_signal(name, content_matrix, inter_matrix, val_inter, u2i, i2i, cf_fn, base_recall):
    cs = make_content_scores(content_matrix, inter_matrix)
    solo = evaluate_score_fn(cs, val_inter, inter_matrix, u2i, i2i)
    best = (0, None)
    for alpha in [0.3, 0.5, 0.7, 0.9]:
        m = evaluate_score_fn(make_hybrid(cf_fn, cs, alpha), val_inter, inter_matrix, u2i, i2i)
        if m['Recall@K'] >= best[0]:
            best = (m['Recall@K'], alpha)
    if best[1] is None:
        best = (0.0, 0.5)   # если все alpha дали 0 то берём середину
    pop = np.asarray(inter_matrix.sum(axis=0)).flatten()
    tail = set(np.argsort(-pop)[int(0.2 * inter_matrix.shape[1]):].tolist())
    h_tail = evaluate_score_fn(make_hybrid(cf_fn, cs, best[1]), val_inter, inter_matrix, u2i, i2i,
                               item_idx_filter=tail)
    b_tail = evaluate_score_fn(cf_fn, val_inter, inter_matrix, u2i, i2i, item_idx_filter=tail)
    gain = (best[0] / base_recall - 1) * 100
    gain_tail = ((h_tail['Recall@K'] / b_tail['Recall@K'] - 1) * 100
                 if b_tail['Recall@K'] > 0 else float('nan'))
    print(f"{name:<14} {solo['Recall@K']:>8.4f} {best[0]:>8.4f} {best[1]:>6} "
          f"{gain:>+9.1f}% {gain_tail:>+11.1f}%")
    return dict(solo=solo['Recall@K'], hybrid=best[0], alpha=best[1],
                gain=gain, gain_tail=gain_tail)

In [124]:
# baseline nowplaying-RS
base_np = evaluate_score_fn(als_scores, val_interactions, sparse_matrix, user_to_idx, item_to_idx)
print(f"ALS baseline (nowplaying): Recall={base_np['Recall@K']:.4f}")
print(f"{'Сигнал':<14} {'Solo':>8} {'Hybrid':>8} {'alpha':>6} {'прирост':>10} {'прирост tail':>12}")

ALS baseline (nowplaying): Recall=0.0241
Сигнал             Solo   Hybrid  alpha    прирост прирост tail


In [125]:
results = {}
for name, cm in signal_matrices.items():
    results[name] = run_signal(name, cm, sparse_matrix, val_interactions,
                               user_to_idx, item_to_idx, als_scores, base_np['Recall@K'])

text             0.0079   0.0237    0.7      -1.6%        +nan%
audio            0.1025   0.0241    0.9      +0.0%        +nan%
graph            0.0051   0.0254    0.5      +5.4%        +nan%


In [126]:
# мультимодальный свой ALS и свой baseline
vk_als = AlternatingLeastSquares(factors=64, regularization=0.1,
                                 iterations=20, alpha=1.0, random_state=RANDOM_SEED)
vk_als.fit(vk_matrix)

  0%|          | 0/20 [00:00<?, ?it/s]

In [127]:
def vk_als_scores(ui):
    return vk_als.item_factors @ vk_als.user_factors[ui]
base_vk = evaluate_score_fn(vk_als_scores, vk_val_interactions, vk_matrix,
                            vk_user_to_idx, vk_item_to_idx)
print(f"ALS baseline (VK-LSVD): Recall={base_vk['Recall@K']:.4f}")

ALS baseline (VK-LSVD): Recall=0.0200


In [128]:
results['multimodal(VK)'] = run_signal('multimod.(VK)', multi_matrix, vk_matrix,
                                       vk_val_interactions, vk_user_to_idx, vk_item_to_idx,
                                       vk_als_scores, base_vk['Recall@K'])
# Сигналы сравниваем по колонкам "прирост" и "прирост tail"
# они нормированы на собственный baseline и потому сопоставимы между датасетами.

multimod.(VK)    0.0030   0.0226    0.7     +13.0%        -6.4%


## 6. Важность сигналов: permutation importance

Второй от рекомендаций способ измерить информативность сигнала: обучаем
вспомогательную модель предсказывать вес взаимодействия сразу из всех
сигналов, а затем случайно перемешиваем каждый блок признаков и смотрим,
без какого блока качество предсказания падает сильнее, тот сигнал и информативнее.

Метод работает только для сигналов одного датасета, поэтому здесь три блока nowplaying-RS, по 8 первых компонент каждого, иначе крупный блок
«выигрывал» бы просто числом признаков. Мультимодальный сигнал существует на другом каталоге (VK-LSVD) не входит в этот анализ, его информативность показана в выше в разделе 5: прирост +21% к ALS.

In [130]:
# permutation importance по блокам сигналов ==========
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

N_COMP, N_SAMPLE = 8, 200_000
# берём выборку взаимодействий nowplaying-RS и для каждого признаки трёх сигналов
samp = train_w.sample(n=min(N_SAMPLE, len(train_w)), seed=RANDOM_SEED)
iidx = samp['track_id'].replace(item_to_idx).to_numpy()

In [131]:
# samp — это DataFrame с колонками ['user_id', 'track_id', 'weight']
# item_to_idx — словарь {track_id: index}

# Преобразуем track_id в целочисленные индексы
iidx = np.array([item_to_idx[t] for t in samp['track_id']], dtype=np.int64)

blocks, X_parts, feat_names = {}, [], []
for name, cm in signal_matrices.items():
    n = min(N_COMP, cm.shape[1])
    X_parts.append(cm[iidx, :n])
    cols = [f'{name}_{i}' for i in range(n)]
    blocks[name] = cols
    feat_names += cols

X = np.hstack(X_parts).astype(np.float32)
y = np.log1p(samp['weight'].to_numpy()).astype(np.float32)   # цель это вес взаимодействия

print('X shape:', X.shape)
print('y shape:', y.shape)

X shape: (200000, 24)
y shape: (200000,)


In [132]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)
gbm = HistGradientBoostingRegressor(max_iter=150, random_state=RANDOM_SEED).fit(Xtr, ytr)
print(f'R^2 на отложенной выборке: {gbm.score(Xte, yte):.3f}')

R^2 на отложенной выборке: 0.130


In [133]:
pi = permutation_importance(gbm, Xte[:20000], yte[:20000], n_repeats=3, random_state=RANDOM_SEED)
imp = dict(zip(feat_names, pi.importances_mean))
print('\n--- Важность по блокам сигналов (сумма permutation importance) ---')


--- Важность по блокам сигналов (сумма permutation importance) ---


In [134]:
for b, cols_ in sorted(blocks.items(), key=lambda kv: -sum(imp[c] for c in kv[1])):
    print(f'{b:<10} {sum(imp[c] for c in cols_):.4f}')

text       0.1461
graph      0.0995
audio      0.0004


## Вывод

В работе на двух открытых датасетах сопоставлены четыре типа контентных сигналов — текстовый (хэштеги через TF-IDF+SVD), аудио (характеристики Spotify), графовый (item2vec по графу прослушиваний) и мультимодальный (готовый эмбеддинг VK-LSVD, обученный совместно на кадрах, аудио и описании). Каждый сигнал оценён в двух ролях: как самостоятельная контентная модель и как добавка к коллаборативной фильтрации (ALS), с подбором коэффициента смешивания α и отдельной проверкой на сегменте малопопулярных объектов.

**Методическое замечание.** Абсолютные метрики между датасетами несопоставимы (разные каталоги и домены — музыка и видео), поэтому типы сигналов сравнивались по нормированным величинам: приросту гибрида относительно собственного ALS-baseline и его аналогу на «хвосте» непопулярных объектов.

Основные результаты следующие.

**Полезность сигнала определяется не его самостоятельной силой, а независимостью от коллаборативного сигнала.** Графовый сигнал, выученный из того же поведения, что и ALS, силён как самостоятельная модель, но как добавка к ALS даёт незначительный прирост — два поведенческих сигнала дублируют друг друга. Напротив, текстовый и аудио-сигналы слабы «соло», но комплементарны коллаборативной фильтрации: они описывают объект с той стороны, которую поведение не отражает (тематика и звучание), и потому дают заметный прирост в гибриде.

**Контентные сигналы наиболее ценны в сценарии холодного старта.** На «хвосте» малопопулярных объектов, где у коллаборативной фильтрации мало данных для выучивания латентных факторов, добавление контентного сигнала даёт наибольший относительный прирост — контентный эмбеддинг доступен объекту независимо от числа его взаимодействий.

**Мультимодальный сигнал — наиболее богатый среди контентных.** Готовый эмбеддинг VK-LSVD, объединяющий несколько модальностей в одном векторе, дал устойчивый прирост +21% Recall@10 к ALS в основной части проекта и служит эталонной планкой, относительно которой оцениваются одиночные модальности: объединённое представление покрывает больше аспектов объекта, чем любая отдельная модальность.

**Выводы подтверждены двумя независимыми методами.** Ранжирование сигналов по приросту качества в рекомендациях согласуется с их ранжированием по информативности, полученным через permutation importance на вспомогательной задаче предсказания веса взаимодействия, — что повышает надёжность выводов.

**Итог по сценариям.** Контентные сигналы (текстовый, аудио, мультимодальный) наиболее полезны как дополнение к коллаборативной фильтрации и особенно на малопопулярных объектах; поведенческо-графовый сигнал полезен там, где коллаборативная модель недоступна или слишком дорога, но как добавка к ней избыточен. Для промышленной рекомендательной системы это означает: обогащать коллаборативную модель следует сигналами, ортогональными поведению (содержание объекта), а не его производными.

**Ограничения исследования.** Текстовый сигнал построен на социальных тегах, которые шумнее редакционных описаний; аудио-сигнал представлен девятью обобщёнными характеристиками, а не полноценным нейросетевым эмбеддингом звука; мультимодальный сигнал измерен на другом домене (видео) и сопоставляется с остальными только через нормированный прирост, поэтому различие доменов остаётся неустранимым фактором при интерпретации.
